# TP 4 — SVD : de la matrice de cooccurrence aux vecteurs denses

## Rappels

La SVD décompose une matrice $M \in \mathbb{R}^{m \times n}$ en $M = U \Sigma V^{\top}$, où $U$ et $V$ ont des colonnes orthonormées et $\Sigma$ est diagonale, avec $\sigma_1 \geq \sigma_2 \geq \dots \geq 0$.

`numpy` propose une implémentation de l'opération de la `svd`:

```python
U, S, Vt = np.linalg.svd(M)     # attention : V est renvoyé TRANSPOSÉ
M_reconstruite = U @ np.diag(S) @ Vt
```

- On peut approximer $M$ avec un rang $d$ comme suit : `M ≈ (U[:, :d] * S[:d]) @ Vt[:d, :]`.

- Les **représentations denses des mots** sont les lignes de $U_d \Sigma_d$.

- Variance expliquée : $\text{proportion conservée}(d) = \frac{\sum_{i=1}^{d} \sigma_i^2}{\sum_i \sigma_i^2}$.

- $d$ est un **hyperparamètre** : une valeur fixée avant l'exécution, choisie de manière empirique.



# Récupération des données et du code précédent

Exécutez ces cellules qui vous permettent de récupérer données et code du TP précédent. Vous devez obtenir **112 699 paires distinctes** et deux matrices de forme (2000, 2000).

In [1]:
import re
from collections import Counter
from pathlib import Path
from urllib.request import Request, urlopen
import numpy as np

url = "https://www.gutenberg.org/cache/epub/14155/pg14155.txt"
fichier = Path("corpus/madame_bovary_gutenberg_14155.txt")
if not fichier.exists():
    fichier.parent.mkdir(parents=True, exist_ok=True)
    requete = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(requete) as reponse:
        contenu = reponse.read().decode("utf-8-sig")
    fichier.write_text(contenu, encoding="utf-8")

texte_brut = fichier.read_text(encoding="utf-8")
marque_debut = "*** START OF THE PROJECT GUTENBERG EBOOK MADAME BOVARY ***"
marque_fin = "*** END OF THE PROJECT GUTENBERG EBOOK MADAME BOVARY ***"
texte = texte_brut.split(marque_debut, 1)[1].split(marque_fin, 1)[0]

motif_francais = r"[^\W\d_]+(?:['’-][^\W\d_]+)*"
tokens = re.findall(motif_francais, texte.lower())

frequences = Counter(tokens)
vocabulaire = [mot for mot, compte in frequences.most_common(2000)]
ensemble_vocabulaire = set(vocabulaire)
mot_vers_id = {mot: i for i, mot in enumerate(vocabulaire)}
id_vers_mot = {i: mot for mot, i in mot_vers_id.items()}

print(f"{len(tokens):,} tokens, vocabulaire restreint : {len(vocabulaire)}")


111,209 tokens, vocabulaire restreint : 2000


In [2]:
def cooccurrences(tokens, k, vocabulaire):
    ensemble = set(vocabulaire)
    comptes = Counter()
    for i, cible in enumerate(tokens):
        if cible not in ensemble:
            continue
        debut = max(0, i - k)
        fin = min(len(tokens), i + k + 1)
        for j in range(debut, fin):
            if i != j:
                contexte = tokens[j]
                if contexte in ensemble:
                    comptes[(cible, contexte)] += 1
    return comptes


def vers_matrice(comptes, mot_vers_id):
    n = len(mot_vers_id)
    matrice = np.zeros((n, n), dtype=np.int32)
    for (cible, contexte), compte in comptes.items():
        matrice[mot_vers_id[cible], mot_vers_id[contexte]] = compte
    return matrice


def calculer_ppmi(matrice):
    matrice = matrice.astype(float)
    total = matrice.sum()
    sommes_lignes = matrice.sum(axis=1)
    sommes_colonnes = matrice.sum(axis=0)
    effectifs_attendus = np.outer(sommes_lignes, sommes_colonnes) / total
    ppmi = np.zeros_like(matrice)
    masque = matrice > 0
    ppmi[masque] = np.log2(matrice[masque] / effectifs_attendus[masque])
    ppmi = np.maximum(ppmi, 0)
    return ppmi


comptes_bovary = cooccurrences(tokens,
                               2,
                               vocabulaire)
matrice_brute = vers_matrice(comptes_bovary, mot_vers_id)
matrice_ppmi = calculer_ppmi(matrice_brute)

print("paires distinctes :", len(comptes_bovary))
print("matrice brute :", matrice_brute.shape, matrice_brute.dtype)
print("matrice PPMI :", matrice_ppmi.shape, matrice_ppmi.dtype)

paires distinctes : 112699
matrice brute : (2000, 2000) int32
matrice PPMI : (2000, 2000) float64


# Exercice 1 — Taille de la matrice de co-occurence

La séance identifie deux limites à la matrice de cooccurrence:
1. sa **dimension énorme**
2. son **caractère creux**.

**Question 1.** 
1. Combien de cellules contient `matrice_brute` ? 
2. Quelle place occupent en mémoire `matrice_brute` (int32) et `matrice_ppmi` (float64) (utilisez l'attribut `nbytes`) ?


**Question 2.** 
1. Quelle **proportion de cellules nulles** contient `matrice_brute` (utilisez `np.count_nonzero` et la taille de la matrice) ? 
2. En moyenne, combien de contextes non nuls possède un mot du vocabulaire ? 


# Exercice 2 — Exemple d'application de SVD

L'exercice permet de tester la décomposition $M = U \Sigma V^{\top}$. 

```python
contextes_mini = ["dort", "ronronne", "mugit", "laboure", "champ"]
mots_mini = ["chat", "chaton", "boeuf", "veau"]

P = np.array([
    [3, 2, 0, 0, 0],   # chat   : contextes "dort", "ronronne"
    [6, 4, 0, 0, 0],   # chaton : mêmes contextes que chat, comptes doublés
    [0, 0, 4, 2, 1],   # boeuf  : contextes "mugit", "laboure", "champ"
    [0, 0, 8, 4, 2],   # veau   : mêmes contextes que boeuf, comptes doublés
])
```

`np.linalg.svd`, qui renvoie **trois objets** :

```python
U, S, Vt = np.linalg.svd(M, full_matrices=False)
```

- `U` : la matrice $U$, de forme (m, m) — une colonne = un axe latent, vu côté mots ;
- `S` : les valeurs singulières $\sigma_1 \geq \sigma_2 \geq \dots$, stockées dans un **simple vecteur** (pas une matrice !) ;
- `Vt` : la matrice $V^{\top}$, **déjà transposée** — une ligne = un axe latent, vu côté contextes.

L'approximation de rang $d$ garde les $d$ premiers éléments de chaque facteur :

```python
M_approx = (U[:, :d] * S[:d]) @ Vt[:d, :]
```

Chaque colonne de `U[:, :d]` est multipliée par sa valeur singulière : c'est exactement $U_d \Sigma_d$.


**Question 1** 
1. Définissez la matrice `P` ci-dessus dans votre cellule. 
2. Sans exécutez, écrivez en commentaire les dimensions que vous prévoyez pour `U`, `S` et `Vt`.
3. Exécutez ensuite `U, S, Vt = np.linalg.svd(P)` et affichez `U.shape`, `S.shape`, `Vt.shape`, en comparant par rapport à vos prédictions.


**Question 2** 
1. Reconstruisez la matrice avec `P_rec = U @ np.diag(S) @ Vt`
2. Vérifiez l'égalité avec `np.allclose(P_rec, P)`. 

In [40]:
P_rec = U @ np.diag(S) @ Vt

In [41]:
np.allclose(P_rec, P)

True

**Question 3.** On ne garde que la première composante.
1. Construisez `P1 = (U[:, :1] * S[0]) @ Vt[:1, :]` et affichez `np.round(P1, 2)`. 
2. Que restent-il des lignes de `chat` et de `chaton` ?


# Exercice 3 — Projection de Madame Bovary
On applique maintenant la SVD à `matrice_ppmi`, calculée sur Madame Bovary. L'objectif du cours est de transformer des vecteurs **creux** de dimension 2 000 en vecteurs **denses** de dimension $d \ll 2000$ qui conservent les proximités sémantiques.

**Question 1.** 
1. Calculez la SVD complète de `matrice_ppmi`. 
2. Affichez les formes de `U`, `S`, `Vt`, puis les dix premières valeurs singulières arrondies.


**Question 2.** 
1. Écrivez une fonction `variance_expliquee(S, d)` qui renvoie la **proportion d'information conservée** (formule du cours).
2. Affichez un tableau `d → variance expliquée` pour $d \in \{10, 50, 100, 200, 500, 1000, 2000\}$. Quel est le plus petit d qui conserve 80 % de l'information ?


**Question 3.** 
1. Écrivez une fonction `representer(U, S, d)` qui renvoie la matrice des représentations denses des mots (*i.e.* les lignes de $U_d \Sigma_d$, de forme (2000, d)).
2. Pour `d = 100`, quelle proportion de coefficients non nuls contient cette matrice ? Comparez au taux mesuré à l'exercice 1. La représentation est-elle creuse ou dense ?
3. Quelle place occupe-t-elle en mémoire ?

**Question 4.**
1. Comparez, pour `emma`, `charles`, `maison`, `amour` et `cheval`, les dix plus proches voisins dans l'espace PPMI complet (cf TP 3) et dans l'espace réduit à `d = 100` (réutilisez `plus_proches_voisins`).
2. Pour quels mots les voisins changent-ils le plus ? 
3. Comparez le cosinus entre `emma`/`charles` dans la matrice comprimée et la matrice PPMI.


# Exercice 4 — Impact du paramètre $d$

$d$ est un **hyperparamètre** : trop petit, l'espace est sur-comprimé ; trop grand, on garde le bruit. On mesure ici ce compromis directement sur les plus proches voisins.

**Question 1.** 
1. Écrivez une fonction `jaccard(liste1, liste2)` qui renvoie la taille de l'intersection divisée par la taille de l'union de deux listes de mots.
2. Pour chaque $d \in \{10, 50, 100, 300, 1000\}$, affichez la similarité de Jaccard entre les dix voisins de `maison` dans l'espace réduit et les dix voisins de `maison` dans l'espace PPMI complet.
3. Faîtes l'analyse de la variance pour proposer une valeur de $d$.

## Pour aller plus loin:

- Pour aller plus loin sur la **LSA** (*Latent Semantic Analysis*): comparez-la au chapitre 6 de Jurafsky & Martin, *Speech and Language Processing*.
- Visualisez : avec `matplotlib`, tracez un nuage de points de `W[:, 0]` en fonction de `W[:, 1]` et annotez les cinq mots témoins (`plt.annotate`).
- Refaites la projection à partir de `matrice_brute` au lieu de `matrice_ppmi` : la pondération PPMI change-t-elle les voisins de l'espace réduit ?
- Pour de très grandes matrices, on n'utilise pas la SVD complète mais `scipy.sparse.linalg.svds`, qui ne calcule que les $d$ premières composantes.
